# DataMate Multi-Stage Code-Generation Pipeline — Google Colab Pro
**Cấu trúc 6 Cell Module Hóa Độc Lập**:
- **Cell 1**: Kết nối Google Drive, kéo code mới nhất từ GitHub & nạp dữ liệu vào SSD Colab.
- **Cell 2**: Khởi động vLLM Server (chạy nền, tối ưu khởi động ~45s).
- **Cell 3**: Chạy Benchmark 300 câu mẫu (Train Sample) -> Tự động lưu kết quả vào `/MyDrive/zaloAI/ketqua/`.
- **Cell 4**: Chạy Benchmark 3.000 câu đầy đủ (Train Full) -> Tự động lưu kết quả vào `/MyDrive/zaloAI/ketqua/`.
- **Cell 5**: Chạy 1.000 câu thi Public Test -> Tự động lưu file nộp bài vào `/MyDrive/zaloAI/ketqua/`.
- **Cell 6**: Tắt vLLM Server & Giải phóng VRAM GPU (tùy chọn ngắt runtime để tiết kiệm 100% CU).

## 📁 Cell 1: Kết nối Drive, Cập nhật Code từ GitHub & Nạp Data vào SSD Colab
- Tự động `git clone` hoặc `git pull` code mới nhất từ GitHub (`25520969-netizen/Datamate`).
- Đồng bộ trực tiếp thư mục dữ liệu từ Google Drive (`/MyDrive/zaloAI/data`) vào ổ cứng SSD cục bộ của Colab (không cần giải nén zip).

In [ ]:
from google.colab import drive
import os, shutil, zipfile
from pathlib import Path

# 1. Kết nối Google Drive
print("🔗 [1/4] Đang kết nối Google Drive...")
drive.mount("/content/drive")
WORK_DIR = Path("/content/Datamate")

# 2. Tự động kéo Code mới nhất từ GitHub (Mất ~0.5s)
print("
🐙 [2/4] Đang cập nhật mã nguồn mới nhất từ GitHub...")
if not WORK_DIR.exists():
    !git clone https://github.com/25520969-netizen/Datamate.git $WORK_DIR
    %cd $WORK_DIR
else:
    %cd $WORK_DIR
    !git pull

# 3. Đồng bộ dữ liệu từ Google Drive vào SSD Colab (Không cần zip)
print("
📂 [3/4] Đang kiểm tra và đồng bộ dữ liệu vào SSD Colab...")
drive_data_roots = [
    Path("/content/drive/MyDrive/zaloAI/data"),
    Path("/content/drive/MyDrive/zaloAI"),
    Path("/content/drive/MyDrive/data")
]
drive_data_dir = next((p for p in drive_data_roots if (p / "train" / "data").exists()), None)

if drive_data_dir is not None:
    for sub in ["train", "public_test"]:
        src_sub = drive_data_dir / sub
        dst_sub = WORK_DIR / sub
        if src_sub.exists() and not (dst_sub / "data").exists():
            print(f"  -> Đang sao chép thư mục '{sub}' từ Drive sang SSD Colab...")
            shutil.copytree(src_sub, dst_sub, dirs_exist_ok=True)
            
    # Tự động đồng bộ Parquet Cache nếu có trên Drive (giúp Colab nạp 30 shop trong 1.5s thay vì 10 phút)
    cache_candidates = [
        drive_data_dir / ".cache",
        drive_data_dir.parent / ".cache",
        Path("/content/drive/MyDrive/zaloAI/.cache")
    ]
    found_cache = next((c for c in cache_candidates if (c / "ingested").exists()), None)
    if found_cache and not (WORK_DIR / ".cache" / "ingested").exists():
        print(f"  -> ⚡ Phát hiện Parquet cache trên Drive ({found_cache}). Đang đồng bộ siêu tốc...")
        shutil.copytree(found_cache, WORK_DIR / ".cache", dirs_exist_ok=True)
        print("  -> ✅ Đã đồng bộ Parquet cache! Tải dữ liệu toàn bộ shop chỉ mất ~1.5 giây.")
        
    print("✅ Dữ liệu Train & Public Test đã sẵn sàng trên SSD Colab!")
else:
    # Hỗ trợ dự phòng nếu người dùng vẫn dùng file zip
    zip_candidates = [
        Path("/content/drive/MyDrive/zaloAI/zalo_data.zip"),
        Path("/content/drive/MyDrive/zalo_data.zip"),
        Path("/content/drive/MyDrive/zalo_AI.zip")
    ]
    found_zip = next((z for z in zip_candidates if z.exists()), None)
    if found_zip and not (WORK_DIR / "train" / "data").exists():
        print(f"  -> Phát hiện file zip: {found_zip}. Đang giải nén vào SSD Colab...")
        with zipfile.ZipFile(found_zip, "r") as zf:
            zf.extractall(WORK_DIR)
        print("✅ Đã nạp dữ liệu từ file zip thành công!")
    elif not (WORK_DIR / "train" / "data").exists():
        raise FileNotFoundError(
            "Chưa tìm thấy thư mục dữ liệu trên Google Drive!\n"
            "Vui lòng tạo thư mục 'zaloAI/data/' trên MyDrive và tải 2 thư mục 'train' và 'public_test' lên đó."
        )

# 4. Cài đặt các thư viện cần thiết
print("
📦 [4/4] Cài đặt thư viện thực thi...")
!pip install -q vllm duckdb openpyxl pyarrow pytest
print("\n🎉 Bước 1 hoàn tất 100%! Bạn có thể chuyển sang Cell 2 để bật vLLM.")


## ⚡ Cell 2: Khởi động vLLM Server (GPU Continuous Batching)
- Khởi chạy vLLM ở tiến trình nền với cờ tối ưu: `--enforce-eager` (khởi động sau ~45s, không mất 4 phút compile CUDA graph) và `--max-model-len 4096`.

In [ ]:
import subprocess, time, os

print("⏳ Đang khởi động vLLM Server trên GPU Colab...")
# Dọn dẹp tiến trình cũ nếu có
os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")

# Ghi toàn bộ log vLLM vào file vllm.log để theo dõi trực quan
log_file = open("vllm.log", "w")

vllm_proc = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
    "--port", "8000",
    "--max-model-len", "8192",  # 8192 an toàn tuyệt đối trên cả GPU T4 (16GB), L4 (24GB) và A100 (40GB)
    "--gpu-memory-utilization", "0.90",
    "--enable-prefix-caching",
    "--enforce-eager"
], stdout=log_file, stderr=subprocess.STDOUT)

start_t = time.time()
print("🚀 Tiến trình vLLM đã khởi chạy nền. Đang theo dõi tiến trình...")

while True:
    # 1. Phát hiện ngay lập tức nếu vLLM bị dừng/crash (ví dụ thiếu VRAM)
    if vllm_proc.poll() is not None:
        log_file.flush()
        with open("vllm.log", "r") as f:
            err_content = f.read()
        print("\n❌ vLLM đã dừng bất thường! Chi tiết lỗi từ vllm.log:")
        print("=" * 65)
        print(err_content[-2000:])
        print("=" * 65)
        raise RuntimeError("Khởi động vLLM thất bại. Vui lòng xem thông báo lỗi chi tiết ở trên.")

    # 2. Kiểm tra server đã phản hồi 200 OK
    if os.system("curl -s http://localhost:8000/v1/models > /dev/null") == 0:
        print(f"\n✅ vLLM Server đã sẵn sàng 100% (sau {time.time()-start_t:.1f}s)! Bạn có thể chuyển sang Cell 3 để chạy.")
        break

    time.sleep(3)
    elapsed = int(time.time() - start_t)
    if elapsed % 15 == 0:
        log_file.flush()
        try:
            with open("vllm.log", "r") as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]
            last_msg = lines[-1] if lines else "Đang chuẩn bị tải mô hình..."
            print(f"  [Đã chờ {elapsed}s] {last_msg[:85]}...", flush=True)
        except Exception:
            pass


## 🟢 Cell 3: Chạy Benchmark 300 câu mẫu (Train Sample — ~4 đến 5 phút)
- Tự động kéo code mới nhất từ GitHub.
- Tự động chấm điểm Penalized Accuracy với đáp án chuẩn.
- Tự động lưu file kết quả `predictions_sample300.jsonl` vào thư mục `/MyDrive/zaloAI/ketqua/`.

In [ ]:
import os, shutil
from pathlib import Path

CONCURRENCY = 8  # Số luồng chạy song song trên GPU

# 1. Cập nhật code mới nhất từ GitHub
print("🐙 Đang cập nhật code mới nhất từ GitHub...")
!git pull

# 2. Chạy benchmark 300 câu mẫu
print("\n🚀 Bắt đầu chạy Benchmark 300 câu mẫu (30 doanh nghiệp)...")
!python3 run_pipeline.py \
    --questions train/sample_300_questions.jsonl \
    --gold train/sample_300_answers.jsonl \
    --evaluate \
    --data-root train/data \
    --calendar train/calendar.csv \
    --rulebook train/rulebook.md \
    --output predictions_sample300.jsonl \
    --backend openai \
    --api-base http://localhost:8000/v1 \
    --concurrency $CONCURRENCY \
    --fresh

# 3. Lưu file kết quả về Google Drive: /MyDrive/zaloAI/ketqua/
OUT_DIR = Path("/content/drive/MyDrive/zaloAI/ketqua")
OUT_DIR.mkdir(parents=True, exist_ok=True)
local_file = Path("predictions_sample300.jsonl")
drive_file = OUT_DIR / "predictions_sample300.jsonl"

if local_file.exists():
    shutil.copy(str(local_file), str(drive_file))
    print(f"\n💾 [THÀNH CÔNG] File kết quả đã được lưu tại:\n👉 {drive_file}")


## 🟡 Cell 4: Chạy Benchmark toàn bộ 3.000 câu Train (Train Full — ~35 đến 45 phút)
- Tự động kéo code mới nhất từ GitHub.
- Đánh giá tổng thể độ chính xác trên toàn bộ tập dữ liệu huấn luyện.
- Tự động lưu file kết quả `predictions_train3000.jsonl` vào thư mục `/MyDrive/zaloAI/ketqua/`.

In [ ]:
import os, shutil
from pathlib import Path

CONCURRENCY = 8  # Số luồng chạy song song trên GPU

# 1. Cập nhật code mới nhất từ GitHub
print("🐙 Đang cập nhật code mới nhất từ GitHub...")
!git pull

# 2. Chạy toàn bộ 3.000 câu tập Train
print("\n🚀 Bắt đầu chạy Benchmark toàn bộ 3.000 câu tập Train...")
!python3 run_pipeline.py \
    --questions train/questions.jsonl \
    --gold train/answers.jsonl \
    --evaluate \
    --data-root train/data \
    --calendar train/calendar.csv \
    --rulebook train/rulebook.md \
    --output predictions_train3000.jsonl \
    --backend openai \
    --api-base http://localhost:8000/v1 \
    --concurrency $CONCURRENCY \
    --fresh

# 3. Lưu file kết quả về Google Drive: /MyDrive/zaloAI/ketqua/
OUT_DIR = Path("/content/drive/MyDrive/zaloAI/ketqua")
OUT_DIR.mkdir(parents=True, exist_ok=True)
local_file = Path("predictions_train3000.jsonl")
drive_file = OUT_DIR / "predictions_train3000.jsonl"

if local_file.exists():
    shutil.copy(str(local_file), str(drive_file))
    print(f"\n💾 [THÀNH CÔNG] File kết quả đã được lưu tại:\n👉 {drive_file}")


## 🔴 Cell 5: Chạy 1.000 câu Public Test (Xuất file nộp bài — ~12 đến 15 phút)
- Tự động kéo code mới nhất từ GitHub.
- Chạy bài thi trên 14 doanh nghiệp (`dev_001` -> `dev_014`).
- Tự động lưu file nộp bài `submission_public1000.jsonl` vào thư mục `/MyDrive/zaloAI/ketqua/`.

In [ ]:
import os, shutil
from pathlib import Path

CONCURRENCY = 8  # Số luồng chạy song song trên GPU

# 1. Cập nhật code mới nhất từ GitHub
print("🐙 Đang cập nhật code mới nhất từ GitHub...")
!git pull

# 2. Chạy 1.000 câu Public Test nộp bài thi
print("\n🚀 Bắt đầu chạy 1.000 câu thi Public Test (dev_001 -> dev_014)...")
!python3 run_pipeline.py \
    --questions public_test/questions.jsonl \
    --data-root public_test/data \
    --calendar public_test/calendar.csv \
    --rulebook public_test/rulebook.md \
    --output submission_public1000.jsonl \
    --backend openai \
    --api-base http://localhost:8000/v1 \
    --concurrency $CONCURRENCY \
    --fresh

# 3. Lưu file nộp bài về Google Drive: /MyDrive/zaloAI/ketqua/
OUT_DIR = Path("/content/drive/MyDrive/zaloAI/ketqua")
OUT_DIR.mkdir(parents=True, exist_ok=True)
local_file = Path("submission_public1000.jsonl")
drive_file = OUT_DIR / "submission_public1000.jsonl"

if local_file.exists():
    shutil.copy(str(local_file), str(drive_file))
    print(f"\n💾 [THÀNH CÔNG] File nộp bài đã được lưu tại:\n👉 {drive_file}")


## 🛑 Cell 6: Tắt vLLM Server & Giải phóng VRAM GPU
- Tắt tiến trình vLLM để giải phóng 100% VRAM GPU.
- Tùy chọn ngắt kết nối runtime (`DISCONNECT_RUNTIME = True`) để dừng trừ 100% Compute Units Colab khi bạn nghỉ ngơi.

In [ ]:
import os

# 1. Tắt tiến trình vLLM và giải phóng GPU
print("🛑 Đang tắt vLLM Server và giải phóng VRAM GPU...")
os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")
print("✅ Đã tắt vLLM và giải phóng toàn bộ VRAM GPU!")

# 2. Tùy chọn ngắt kết nối máy ảo Colab để tiết kiệm 100% Compute Units
DISCONNECT_RUNTIME = False  # Đổi thành True nếu bạn muốn tắt hẳn máy ảo để không bị trừ CU

if DISCONNECT_RUNTIME:
    print("\n🎉 Đang ngắt kết nối runtime Colab để bảo vệ Compute Units...")
    from google.colab import runtime
    runtime.unassign()
else:
    print("\n💡 Lưu ý: Máy ảo Colab vẫn đang bật. Nếu bạn không chạy thêm, hãy đổi DISCONNECT_RUNTIME = True rồi bấm chạy lại cell này để ngắt hoàn toàn máy ảo.")
